## 1. Carga de datos

Cargamos los archivos CSV de cada compresor en un diccionario `data`,
donde cada clave es el nombre del compresor y el valor es su DataFrame.

Cada CSV contiene las columnas:
- **Presion**: presión atmosférica en la planta
- **Temperatura**: temperatura ambiente en la planta
- **Frecuencia**: frecuencia de operación del compresor
- **Potencia_Medida**: potencia real medida *(target)*
- **Potencia_Estimada**: estimación lineal del fabricante *(baseline)*

In [ ]:
import os
import pygad

import pandas as pd
import numpy as np
from joblib import Parallel, delayed
from itertools import product

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [ ]:
ruta_base = "Datos/Originales"
archivos = ["CompA.csv", "CompB.csv", "CompC.csv", "CompD.csv"]
data = {}

for archivo in archivos:
    ruta_completa = os.path.join(ruta_base, archivo)
    nombre = archivo.replace(".csv", "")
    data[nombre] = pd.read_csv(ruta_completa)

print(data["CompA"].head())

    Presion  Temperatura  Frecuencia  Potencia_Medida  Potencia_Estimada
0  0.775748         16.9         0.0        71.466562          77.432556
1  0.776315         16.6         0.0        71.442768          77.498194
2  0.776784         16.6         0.0        71.347653          77.477151
3  0.777093         16.4         0.0        71.244807          77.524010
4  0.777436         16.4         0.0        71.194101          77.508609


In [14]:
for nombre, df in data.items():
    df.drop_duplicates(inplace=True)
    
    df = df[df["Frecuencia"] >= 0]
    df = df[df["Frecuencia"] <= 100]
    
    data[nombre] = df

## 2. Entrenamiento de modelos de regresión

Para predecir el consumo real de cada compresor usamos datos históricos medidos en planta.
Entrenamos un modelo de regresión por compresor con las variables `Presion`, `Temperatura` y `Frecuencia` como entrada y `Potencia_Medida` como target.

Comparamos tres modelos:
- **RandomForest**: captura relaciones no lineales entre variables
- **GradientBoosting**: similar a RandomForest pero entrena de forma secuencial, corrigiendo errores
- **Ridge**: regresión lineal regularizada, útil como baseline para comparar con los modelos no lineales

Para evaluar la calidad de cada modelo usamos **validación cruzada con 3 folds**, que nos da el **R²** medio y su desviación estándar (±).
Un R² cercano a 1 indica que el modelo predice bien. El mejor modelo por compresor se selecciona automáticamente.

In [ ]:
def get_modelos_candidatos():
    return {
        "RandomForest":       RandomForestRegressor(n_estimators=10, random_state=42, n_jobs=-1),
        "GradientBoosting":   GradientBoostingRegressor(n_estimators=10, random_state=42),
        "Ridge":              Pipeline([("scaler", StandardScaler()), ("ridge", Ridge())]),
    }

def entrenar_y_evaluar(nombre_comp, df, nombre_modelo, modelo):
    X = df[["Presion", "Temperatura", "Frecuencia"]]
    y = df["Potencia_Medida"]
    modelo.fit(X, y)
    scores = cross_val_score(modelo, X, y, cv=5, scoring="r2", n_jobs=-1)
    return nombre_comp, nombre_modelo, modelo, np.mean(scores), np.std(scores)

tareas = [
    (nombre_comp, df, nombre_modelo, modelo)
    for nombre_comp, df in data.items()
    for nombre_modelo, modelo in get_modelos_candidatos().items()
]

resultados_modelos = Parallel(n_jobs=-1)(
    delayed(entrenar_y_evaluar)(*t) for t in tareas
)

rows = []
for nombre_comp, nombre_modelo, _, r2_mean, r2_std in resultados_modelos:
    rows.append({"Compresor": nombre_comp, "Modelo": nombre_modelo,
                 "R2_medio": round(r2_mean, 4), "R2_std": round(r2_std, 4)})

df_r2 = pd.DataFrame(rows).sort_values(["Compresor", "R2_medio"], ascending=[True, False])
print("\n=== R² por compresor y modelo ===")
print(df_r2.to_string(index=False))

modelos = {}
for nombre_comp in data.keys():
    candidatos = [(r2, mod, nm) for (nc, nm, mod, r2, _) in resultados_modelos if nc == nombre_comp]
    best_r2, best_mod, best_name = max(candidatos, key=lambda x: x[0])
    modelos[nombre_comp] = best_mod
    print(f"{nombre_comp} → mejor modelo: {best_name} (R²={best_r2:.4f})")


=== R² por compresor y modelo ===
Compresor           Modelo  R2_medio  R2_std
    CompA     RandomForest    0.9592  0.0219
    CompA            Ridge    0.9387  0.0287
    CompA GradientBoosting    0.8346  0.0234
    CompB     RandomForest    0.9876  0.0029
    CompB            Ridge    0.9612  0.0058
    CompB GradientBoosting    0.8495  0.0206
    CompC     RandomForest    0.9852  0.0027
    CompC            Ridge    0.9544  0.0083
    CompC GradientBoosting    0.8459  0.0089
    CompD     RandomForest    0.9874  0.0016
    CompD            Ridge    0.9611  0.0105
    CompD GradientBoosting    0.8469  0.0173
CompA → mejor modelo: RandomForest (R²=0.9592)
CompB → mejor modelo: RandomForest (R²=0.9876)
CompC → mejor modelo: RandomForest (R²=0.9852)
CompD → mejor modelo: RandomForest (R²=0.9874)


In [34]:
# ─────────────────────────────────────────────
# 2. CONFIGURACIONES DEL GA A COMPARAR
# ─────────────────────────────────────────────

configs_ga = {
    "GA_base": dict(
        num_generations=50,       
        sol_per_pop=30,           
        num_parents_mating=6,
        crossover_type="single_point",
        mutation_type="random", mutation_num_genes=1,
        keep_elitism=1, parent_selection_type="sss",
    ),
    "GA_uniform": dict(
        num_generations=100,      
        sol_per_pop=50,           
        num_parents_mating=10,
        crossover_type="uniform",
        mutation_type="random", mutation_num_genes=2,
        keep_elitism=2, parent_selection_type="tournament",
    ),
}

In [35]:
# ─────────────────────────────────────────────
# 3. CASOS DE PRUEBA
# ─────────────────────────────────────────────

casos = [
    {"nombre": "Caso1_normal",    "P": 0.78, "T": 16.5, "caudal_objetivo": 250, "freqs_previas": {"CompA":50,"CompB":50,"CompC":50,"CompD":50}},
    {"nombre": "Caso2_calor",     "P": 0.80, "T": 30.0, "caudal_objetivo": 320, "freqs_previas": {"CompA":70,"CompB":60,"CompC":65,"CompD":70}},
    {"nombre": "Caso3_frio",      "P": 0.75, "T":  5.0, "caudal_objetivo": 150, "freqs_previas": {"CompA":30,"CompB":30,"CompC":30,"CompD":30}},
    {"nombre": "Caso4_maximo",    "P": 0.82, "T": 35.0, "caudal_objetivo": 380, "freqs_previas": {"CompA":80,"CompB":80,"CompC":80,"CompD":80}},
]

caudales = {"CompA": 100, "CompB": 90, "CompC": 95, "CompD": 110}
nombres   = ["CompA", "CompB", "CompC", "CompD"]

In [36]:
# ─────────────────────────────────────────────
# 4. EXPERIMENTO COMPLETO
# ─────────────────────────────────────────────

def correr_experimento(caso, nombre_config, config):
    P              = caso["P"]
    T              = caso["T"]
    caudal_obj     = caso["caudal_objetivo"]
    freqs_previas  = caso["freqs_previas"]

    def fitness_func(ga_instance, solution, solution_idx):
        freqs = dict(zip(nombres, solution))
        consumo_total = 0

        for comp in nombres:
            entrada = pd.DataFrame([[P, T, freqs[comp]]],
                                   columns=["Presion", "Temperatura", "Frecuencia"])
            consumo_total += modelos[comp].predict(entrada)[0]

        cambios = sum(1 for c in nombres if abs(freqs[c] - freqs_previas[c]) > 0.01)
        consumo_total += cambios * 3

        penalizacion = 0
        caudal = sum((freqs[c] / 100) * caudales[c] for c in nombres)
        if caudal < caudal_obj:
            penalizacion += 1000 * (caudal_obj - caudal)
        for f in solution:
            if f > 90:
                penalizacion += 500 * (f - 90)
            if f < 5:
                penalizacion += 500

        return -(consumo_total + penalizacion)

    ga = pygad.GA(
        **config,
        fitness_func=fitness_func,
        num_genes=4,
        gene_space=[{"low": 5, "high": 100}] * 4,
        suppress_warnings=True,
    )
    ga.run()

    sol, fit, _ = ga.best_solution()
    freqs_opt   = dict(zip(nombres, sol))
    caudal      = sum((freqs_opt[c] / 100) * caudales[c] for c in nombres)
    consumo     = 0
    for comp in nombres:
        entrada = pd.DataFrame([[P, T, freqs_opt[comp]]],
                               columns=["Presion", "Temperatura", "Frecuencia"])
        consumo += modelos[comp].predict(entrada)[0]

    return {
        "Caso":          caso["nombre"],
        "Config_GA":     nombre_config,
        "Fitness":       round(-fit, 2),
        "Consumo_W":     round(consumo, 2),
        "Caudal_ls":     round(caudal, 2),
        "Objetivo_ls":   caudal_obj,
        "Cumple":        "✓" if caudal >= caudal_obj else "✗",
        **{f"Freq_{c}": round(freqs_opt[c], 1) for c in nombres},
    }

# Ejecuta todos los casos x todas las configs
resultados_ga = []
for caso in casos:
    for nombre_config, config in configs_ga.items():
        print(f"  Corriendo {caso['nombre']} con {nombre_config}...")
        res = correr_experimento(caso, nombre_config, config)
        resultados_ga.append(res)

  Corriendo Caso1_normal con GA_base...
  Corriendo Caso1_normal con GA_uniform...
  Corriendo Caso2_calor con GA_base...
  Corriendo Caso2_calor con GA_uniform...
  Corriendo Caso3_frio con GA_base...
  Corriendo Caso3_frio con GA_uniform...
  Corriendo Caso4_maximo con GA_base...
  Corriendo Caso4_maximo con GA_uniform...


In [37]:
# ─────────────────────────────────────────────
# 5. TABLA DE RESULTADOS
# ─────────────────────────────────────────────

df_res = pd.DataFrame(resultados_ga)
print("\n=== RESULTADOS COMPLETOS ===")
print(df_res[["Caso","Config_GA","Cumple","Caudal_ls","Objetivo_ls",
              "Consumo_W","Freq_CompA","Freq_CompB","Freq_CompC","Freq_CompD"]].to_string(index=False))

print("\n=== MEJOR CONFIG POR CASO (menor consumo que cumple) ===")
df_validos = df_res[df_res["Cumple"] == "✓"]
if not df_validos.empty:
    idx_mejor = df_validos.groupby("Caso")["Consumo_W"].idxmin()
    print(df_validos.loc[idx_mejor][["Caso","Config_GA","Consumo_W","Caudal_ls"]].to_string(index=False))
else:
    print("Ninguna configuración cumplió el caudal objetivo en todos los casos.")


=== RESULTADOS COMPLETOS ===
        Caso  Config_GA Cumple  Caudal_ls  Objetivo_ls  Consumo_W  Freq_CompA  Freq_CompB  Freq_CompC  Freq_CompD
Caso1_normal    GA_base      ✓     251.12          250     445.41        89.2        71.5         7.1        82.5
Caso1_normal GA_uniform      ✓     250.46          250     445.72        87.6         5.3        64.4        88.1
 Caso2_calor    GA_base      ✓     320.14          320     487.96        87.9        80.9        63.6        89.9
 Caso2_calor GA_uniform      ✓     328.12          320     490.26        88.7        89.0        64.7        89.0
  Caso3_frio    GA_base      ✓     158.71          150     398.54        51.0         5.6         5.1        88.9
  Caso3_frio GA_uniform      ✓     150.87          150     401.50        44.9         5.2        12.0        81.7
Caso4_maximo    GA_base      ✓     380.09          380     502.77        96.6        91.3        97.1        99.2
Caso4_maximo GA_uniform      ✓     381.85          380    